# try19res_py12 — 複数シード一括学習 (F-6)

余っている GPU を使い、**複数シードを並列学習**して seed アンサンブル用のモデル群を作る。
checkpoint 間のばらつき（120k当たり/140k外れ）対策の本命＝独立モデルの多数決。

**重要 — 並列の真の制約は GPU ではなく System RAM（リプレイバッファ）**
- `buffer_size` × 観測(130×29×4B) がそのまま RAM になる。20万だと約6GB/run。
- Colab の System RAM は約12.7GBなので、**フルバッファだと1〜2本が限界**。
- 並列したいなら `--buffer-size` を小さく（例 3万〜5万）＋ `--optimize-memory`。

**手順**: GPU確認 → clone → 依存 → 並列学習 → モデルを file.io でDL

## 0. GPU / RAM 確認

In [ ]:
!nvidia-smi
import psutil
print('CPU コア:', psutil.cpu_count(), '| System RAM(GB):', round(psutil.virtual_memory().total/1e9, 1))

## 1. コード取得（GitHub clone）
プライベートrepoなので token を使う。token は GitHub → Settings → Developer settings →
Fine-grained token で、このrepoの **Contents: Read** だけ付ければよい。

In [ ]:
import getpass
TOKEN = getpass.getpass('GitHub token: ')
REPO = 'github.com/Waipy252/resnet-dqn.git'
%cd /content
!rm -rf resnet-dqn
!git clone --depth 1 https://{TOKEN}@{REPO}
%cd /content/resnet-dqn/try19res_py12
!pwd && ls

## 2. 依存インストール（torchは触らない / 旧gymは入れない）

In [ ]:
!pip install -q \
    "stable-baselines3==2.8.0" \
    "sb3-contrib==2.8.0" \
    "gymnasium" "shimmy" \
    "yfinance" "pandas-datareader"
print('done')

## 3. 複数シード並列学習
`train_multi.py` が学習データを一度だけDLしてキャッシュ → 各シードを最大 `--jobs` 並列で実行。
各シードのモデルは `models/seed{S}/` に保存される。

**目安**
- `--jobs` は CPU コア数程度（Colab無料は2）。RAMに余裕がなければ 1〜2。
- `--buffer-size` × `--jobs` × 約6GB/20万 が RAM 消費の目安。例: buffer 3万・jobs 2 なら約2GB。
- まず短い `--timesteps`（例 5000）で**動作確認**してから本番（15万など）。

In [ ]:
# --- 動作確認（短時間）---
!python train_multi.py --seeds 0 1 --jobs 2 --timesteps 5000 --save-freq 5000 \
    --buffer-size 30000 --optimize-memory
import glob
print('produced:', sorted(glob.glob('models/seed*/*.zip')))

In [ ]:
# --- 本番（例: 5シード × 15万ステップ、2並列）---
# RAM/時間と相談して seeds・jobs・timesteps を調整する
!python train_multi.py --seeds 0 1 2 3 4 --jobs 2 --timesteps 150000 --save-freq 10000 \
    --buffer-size 50000 --optimize-memory

In [ ]:
# 進捗は各シードのログで確認（別セルで実行）
!tail -n 5 train_seed*.log 2>/dev/null; nvidia-smi | head -n 15

## 4. モデルをまとめてダウンロード（Drive不要）
`models/` 以下を1つの tar にまとめ、file.io にアップロード。返る JSON の `"link"` をコピーして
Mac側で `curl -L -o models_multi.tar.gz "<link>" && tar xzf models_multi.tar.gz` する。
> file.io は1回DLで消える。複数回DLしたいときは 0x0.st（コメント参照）。

In [ ]:
!tar czf models_multi.tar.gz models/ && ls -lh models_multi.tar.gz
!curl -F "file=@models_multi.tar.gz" https://file.io
# 複数回DL・30日保持したい場合:
# !curl -F "file=@models_multi.tar.gz" https://0x0.st